In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# Read the data
df = pd.read_csv(r'E:/Desktop/all_data/data8/BjaRmut.csv')

# Extract the required data columns
L = df['LBD'].dropna()
P1 =(df['RPU']).dropna()
I = df['inducer'].dropna()

# 固定参数
I0 = 0.006969286




Imax = 35.84931505



kd =1.13083303




# 使用您的原始函数
def func(x, k1, k2, k3):
    L, I = x
    if not 0 < k1 < 10 or not 0 < k2 or not 0 < k3:
        return np.ones_like(L) * 1e10  # 返回一个大的值
    x1 = np.sqrt((k2*I+1)*(k2*I+1)+ 8 * L * (k2*k2*k3*I*I + k1))
    return (Imax * ((L / 2) * ((x1 - (k2*I+1)) / (x1 + (k2*I+1))) * (kd)/ (1 + (L / 2) * ((x1 - (k2*I+1)) / (x1 + (k2*I+1))) * (kd))) + I0)

# 使用边界约束而不是函数内检查
bounds = ([1e-20, 1e-15, 1e-15], [1000, 1000000000000, 100000000000000000000000])

# 使用 curve_fit 函数拟合模型参数
try:
    params, pcov = curve_fit(func, (L, I), P1, 
                           p0=[0.1, 0.1, 0.1], 
                           bounds=bounds, 
                           maxfev=1000000000)
    
    k1, k2, k3 = params
    print('k1 = %.15f' % k1)
    print('k2 = %.15f' % k2)
    print('k3 = %.15f' % k3)
    
    # 计算拟合曲线
    P1_fit = func((L, I), k1, k2, k3)
    
    # 计算残差平方和
    residuals = P1 - P1_fit
    ssr = np.sum(residuals ** 2)
    print('残差平方和: %.4f' % ssr)
    
    # 计算R²
    ss_tot = np.sum((P1 - np.mean(P1)) ** 2)
    r_squared = 1 - (ssr / ss_tot)
    print('R²: %.4f' % r_squared)
    
except Exception as e:
    print(f"拟合失败: {e}")
    # 如果使用边界约束失败，回退到原始方法
    print("尝试使用无边界约束的拟合...")
    params, pcov = curve_fit(func, (L, I), P1, p0=[0.1, 0.1, 0.1], maxfev=9000000000)
    
    k1, k2, k3 = params
    print('k1 = %.15f' % k1)
    print('k2 = %.15f' % k2)
    print('k3 = %.15f' % k3)
    
    # 计算拟合曲线
    P1_fit = func((L, I), k1, k2, k3)
    
    # 计算残差平方和
    residuals = P1 - P1_fit
    ssr = np.sum(residuals ** 2)
    print('残差平方和: %.4f' % ssr)
    
    # 计算R²
    ss_tot = np.sum((P1 - np.mean(P1)) ** 2)
    r_squared = 1 - (ssr / ss_tot)
    print('R²: %.4f' % r_squared)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# Read the data
df = pd.read_csv(r'E:/Desktop/mamm/414.csv')

# Extract the required data columns
L = df['LBD'].dropna()
P1 = (df['RPU']).dropna()
I = df['inducer'].dropna()

# 固定参数
I0 = 0.064825979



Imax = 13.60910816
kd = 9.152591138





# 精确按照您提供的公式
def func(x, k1, k2, k3, kx1, kx2):
    L, I = x
    
    # 参数边界检查
    if not 0 < k1  or not 0 < k2 or not 0 < k3 or not 0 < kx1 or not 0 < kx2:
        return np.ones_like(L) * 1e10
    
    # 计算 x1
    term = k2*I + kx1 + kx2*k2*I + 1
    x1 = np.sqrt(term*term + 8 * L * (k2*k2*k3*I*I + k1 + k1*kx1*kx1 + k3*k2*k2*kx2*kx2*I*I))
    
    # 计算返回值
    ratio = (x1 - term) / (x1 + term)
    numerator = (L / 2) * ratio * kd
    denominator = 1 + (L / 2) * ratio * kd
    
    return Imax * (numerator / denominator) + I0

# 使用 curve_fit 函数拟合模型参数
params, pcov = curve_fit(func, (L, I), P1, 
                       p0=[0.001, 1, 1000000, 10, 10], 
                       maxfev=1000000000)

k1, k2, k3, kx1, kx2 = params
print('k1 = %.15f' % k1)
print('k2 = %.15f' % k2)
print('k3 = %.15f' % k3)
print('kx1 = %.15f' % kx1)
print('kx2 = %.15f' % kx2)

# 计算拟合曲线
P1_fit = func((L, I), k1, k2, k3, kx1, kx2)

# 计算残差平方和
residuals = P1 - P1_fit
ssr = np.sum(residuals ** 2)
print('残差平方和: %.4f' % ssr)

# 计算R²
ss_tot = np.sum((P1 - np.mean(P1)) ** 2)
r_squared = 1 - (ssr / ss_tot)
print('R²: %.4f' % r_squared)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import os
import glob

# 文件夹路径
folder_path = r"E:/Desktop/mamm/TraR"
file_pattern = os.path.join(folder_path, "*.csv")
file_paths = glob.glob(file_pattern)

# 存储所有数据
all_data = []

# 读取所有文件
for file_path in file_paths:
    df = pd.read_csv(file_path).dropna(subset=["inducer", "LBD", "RPU"])
    df['file_id'] = len(all_data)  # 为每个文件分配唯一ID
    all_data.append(df)

# 合并所有数据
combined_df = pd.concat(all_data, ignore_index=True)

# 提取数据
I = combined_df["inducer"].values
L = combined_df["LBD"].values
y = np.log10(combined_df["RPU"].values)  # 注意：这里不再使用log10转换
file_ids = combined_df["file_id"].values  # 文件标识符

# 常量
I0 = 0.014378964
Imax = 13.60910816

# 使用新公式的固定kd模型函数
def model_func_fixed_kd(x, k1_shared, k2_shared, k3_shared):
    L, I, file_id = x
    
    # 为每个文件分配固定的kd值
    kd_array = np.zeros_like(file_id, dtype=float)
    for i, fid in enumerate(file_id):
        if fid == 0:
            kd_array[i] = 10.56826129  # 第一个文件的kd
        elif fid == 1:
            kd_array[i] = 9.152591138  # 第二个文件的kd
        else:
            kd_array[i] = 2.09409  # 第三个文件的kd
    
    # 新公式
    x1 = np.sqrt((k2_shared*I+1)*(k2_shared*I+1) + 8 * L * (k2_shared*k2_shared*k3_shared*I*I + k1_shared))
    ratio = (x1 - (k2_shared*I+1)) / (x1 + (k2_shared*I+1))
    numerator = (L / 2) * ratio * kd_array
    denominator = 1 + (L / 2) * ratio * kd_array
    
    return np.log10(Imax * (numerator / denominator) + I0)

# 初始猜测值和边界 - 只拟合3个共享参数
p0 = [1, 1, 0.1]
bounds_lower = [0.00000000000000001, 0.00000001, 0.000001]
bounds_upper = [1000000000000, 10000000000, 100000000000]
bounds = (bounds_lower, bounds_upper)

# 拟合固定kd模型
print(f"正在拟合 {len(file_paths)} 个文件（固定kd模式）...")
print("固定kd值：")
for i, file_path in enumerate(file_paths):
    file_name = os.path.basename(file_path)
    if i == 0:
        kd_value = 10.56826129

    elif i == 1:
        kd_value = 9.152591138

    else:
        kd_value = 2.09409

    print(f"{file_name}: kd = {kd_value:.6f}")

try:
    popt_fixed, pcov_fixed = curve_fit(model_func_fixed_kd, (L, I, file_ids), y, 
                                      p0=p0, bounds=bounds, maxfev=500000000)
    
    # 提取参数
    k1_shared, k2_shared, k3_shared = popt_fixed
    
    # 打印共享参数
    print("\n✅ 固定kd拟合的共享参数：")
    print(f"k1_shared = {k1_shared:.15f}")
    print(f"k2_shared = {k2_shared:.6f}")
    print(f"k3_shared = {k3_shared:.6f}")
    
    # 打印固定的kd值
    print("\n📊 固定kd参数：")
    for i, file_path in enumerate(file_paths):
        file_name = os.path.basename(file_path)
        if i == 0:
            kd_value = 10.56826129
        elif i == 1:
            kd_value = 9.152591138

        else:
            kd_value = 2.09409
        print(f"{file_name}: kd = {kd_value:.6f} (固定)")
    
    # 计算总体R²
    y_pred = model_func_fixed_kd((L, I, file_ids), *popt_fixed)
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2_total = 1 - (ss_res / ss_tot)
    print(f"\n📈 总体 R² = {r2_total:.4f}")
    
    # 计算每个文件的R²
    print("\n📈 各文件R²：")
    for i, file_path in enumerate(file_paths):
        file_name = os.path.basename(file_path)
        mask = file_ids == i
        y_file = y[mask]
        y_pred_file = y_pred[mask]
        ss_res_file = np.sum((y_file - y_pred_file)**2)
        ss_tot_file = np.sum((y_file - np.mean(y_file))**2)
        r2_file = 1 - (ss_res_file / ss_tot_file) if ss_tot_file != 0 else 0
        
        if i == 0:
            kd_value = 10.56826129
        elif i == 1:
            kd_value = 9.152591138
        else:
            kd_value = 2.09409
            
        print(f"{file_name}: kd = {kd_value:.6f}, R² = {r2_file:.4f}")
    

except Exception as e:
    print(f"拟合过程中出现错误: {e}")
    print("尝试调整初始参数或边界条件...")
    
    # 提供备选初始参数
    alternative_p0 = [
        [0.0001, 10, 100],
        [0.001, 1, 1000],
        [0.01, 100, 1]
    ]
    
    print("可以尝试的备选初始参数：")
    for i, p0_alt in enumerate(alternative_p0):
        print(f"备选 {i+1}: {p0_alt}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os
import torch.nn.init as init

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.k1 = nn.Parameter(torch.Tensor(1))
        self.k2 = nn.Parameter(torch.Tensor(1))
        self.k3 = nn.Parameter(torch.Tensor(1))
        self.kd = nn.Parameter(torch.Tensor(1))
        init.constant_(self.k1, 0.001)
        init.constant_(self.k2, 0.02735345)
        init.constant_(self.k3, 0.50865889)
        init.constant_(self.kd, 10.0)

    def forward(self, L, I, I0, Imax):
        x1 = torch.sqrt((self.k2*I+1)*(self.k2*I+1)+ 8 * L * (self.k2*self.k2*self.k3*I*I + self.k1)+1e-9)
        P1 = (Imax * ((L / 2 + 1e-9) * ((x1 -  (self.k2*I+1)+ 1e-9) / (x1 +  (self.k2*I+1)) +1e-9) * (self.kd + 1e-9) / (1 + (L / 2 + 1e-9) * ((x1 -  (self.k2*I+1)+ 1e-9) / (x1 +  (self.k2*I+1)+1e-9) + 1e-9) *(self.kd + 1e-9))) + I0)
        return P1

# 初始化模型
model = MyModel()

# 遍历文件夹中的csv文件，将所有的数据合并到一起
folder_path = r'E:/Desktop/data/data1'
data = []
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        # 读取csv文件
        df = pd.read_csv(os.path.join(folder_path, filename)).dropna(subset=["LBD", "inducer", "RPU"])
        data.append(df)

# 合并所有的数据
data_all = pd.concat(data).dropna(subset=["LBD", "inducer", "RPU"])

# 提取你需要的列
L_all = torch.tensor(data_all['LBD'].values, dtype=torch.float32)
I_all = torch.tensor(data_all['inducer'].values, dtype=torch.float32)
I0_all = torch.tensor(0.059199421, dtype=torch.float32)
Imax_all = torch.tensor(13.60910816, dtype=torch.float32)
y_all = torch.tensor(data_all['RPU'].values, dtype=torch.float32)

# 训练模型得到k1, k2, k3
optimizer_all = torch.optim.Adam([model.k1, model.k2, model.k3], lr=0.001)
for _ in range(100000):  # 迭代1000次
    optimizer_all.zero_grad()
    output_all = model(L_all, I_all, I0_all, Imax_all)
    loss_all = nn.MSELoss()(output_all, y_all)
    loss_all.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
    optimizer_all.step()
print(f'Iteration {_}, Loss {loss_all.item()}')
    

print(f'k1={model.k1.item()}, k2={model.k2.item()}, k3={model.k3.item()}')

# 对每个文件单独拟合kd
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        # 读取csv文件
        df = pd.read_csv(os.path.join(folder_path, filename)).dropna(subset=["LBD", "inducer", "RPU"])
        # 提取你需要的列
        L = torch.tensor(df['LBD'].values, dtype=torch.float32)
        I = torch.tensor(df['inducer'].values, dtype=torch.float32)
        I0 = torch.tensor(0.059199421, dtype=torch.float32)
        Imax = torch.tensor(13.60910816, dtype=torch.float32)
        y = torch.tensor(df['RPU'].values, dtype=torch.float32)

        # 训练模型
        optimizer_kd = torch.optim.Adam([model.kd], lr=0.001)
        for _ in range(150000):  
            optimizer_kd.zero_grad()
            output = model(L, I, I0, Imax)
            loss = nn.MSELoss()(output, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            optimizer_kd.step()
        print(f'File {filename}, Iteration {_}, Loss {loss.item()}')
            
        
        print(f'For file {filename}, kd={model.kd.item()}')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os
import torch.nn.init as init

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.k1 = nn.Parameter(torch.Tensor(1))
        self.k2 = nn.Parameter(torch.Tensor(1))
        self.k3 = nn.Parameter(torch.Tensor(1))
        self.kx1 = nn.Parameter(torch.Tensor(1))
        self.kx2 = nn.Parameter(torch.Tensor(1))
        self.kd = nn.Parameter(torch.Tensor(1))
        init.constant_(self.k1, 0.000748235207890)
        init.constant_(self.k2, 0.456403423356008)
        init.constant_(self.k3, 0.023932304995422)
        init.constant_(self.kx1, 0.229368425090218)
        init.constant_(self.kx2, 0.000046218624817)
        init.constant_(self.kd, 10.0)

  
    def forward(self, L, I, I0, Imax):
        # 确保所有参数为正值
        k1 = torch.abs(self.k1)
        k2 = torch.abs(self.k2)
        k3 = torch.abs(self.k3)
        kx1 = torch.abs(self.kx1)
        kx2 = torch.abs(self.kx2)
        kd = torch.abs(self.kd)
        kx2 = torch.clamp(torch.abs(self.kx2), min=0.001)

        x1 = torch.sqrt((k2 * I + kx1 + kx2 * k2 * I + 1) ** 2 + 
                        8 * L * (k2 ** 2 * k3 * I ** 2 + 
                                 k1 + k1 * kx1 ** 2 + 
                                 k3 * k2 ** 2 * kx2 ** 2 * I ** 2))
        P1 = (Imax * ((L / 2) * ((x1 - (k2 * I + kx1 + kx2 * k2 * I + 1)) / 
               (x1 + (k2 * I + kx1 + kx2 * k2 * I + 1))) * kd / 
               (1 + (L / 2) * ((x1 - (k2 * I + kx1 + kx2 * k2 * I + 1)) / 
               (x1 + (k2 * I + kx1 + kx2 * k2 * I + 1))) * kd)) + I0)
        return P1

# 初始化模型
model = MyModel()

# 遍历文件夹中的csv文件，将所有的数据合并到一起
folder_path = 'E:/Desktop/mamm/MR'
data = []
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        # 读取csv文件
        df = pd.read_csv(os.path.join(folder_path, filename)).dropna(subset=["LBD", "inducer", "RPU"])
        data.append(df)

# 合并所有的数据
data_all = pd.concat(data).dropna(subset=["LBD", "inducer", "RPU"])

# 提取需要的列
L_all = torch.tensor(data_all['LBD'].values, dtype=torch.float32)
I_all = torch.tensor(data_all['inducer'].values, dtype=torch.float32)
I0_all = torch.tensor(0.062385854, dtype=torch.float32)
Imax_all = torch.tensor(13.60910816, dtype=torch.float32)
y_all = torch.tensor(data_all['RPU'].values, dtype=torch.float32)

# 训练模型得到k1, k2, k3
optimizer_all = torch.optim.Adam([model.k1, model.k2, model.k3, model.kx1, model.kx2], lr=0.001)
for _ in range(100000):  # 迭代100000次
    optimizer_all.zero_grad()
    output_all = model(L_all, I_all, I0_all, Imax_all)
    loss_all = nn.MSELoss()(output_all, y_all)
    loss_all.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
    optimizer_all.step()
print(f'Iteration {_}, Loss {loss_all.item()}')

print(f'k1={torch.abs(model.k1).item()}, k2={torch.abs(model.k2).item()}, k3={torch.abs(model.k3).item()}, kx1={torch.abs(model.kx1).item()}, kx2={torch.abs(model.kx2).item()}')

# 对每个文件单独拟合kd
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        # 读取csv文件
        df = pd.read_csv(os.path.join(folder_path, filename)).dropna(subset=["LBD", "inducer", "RPU"])
        # 提取你需要的列
        L = torch.tensor(df['LBD'].values, dtype=torch.float32)
        I = torch.tensor(df['inducer'].values, dtype=torch.float32)
        I0 = torch.tensor(0.062385854, dtype=torch.float32)
        Imax = torch.tensor(13.60910816, dtype=torch.float32)
        y = torch.tensor(df['RPU'].values, dtype=torch.float32)

        # 训练模型
        optimizer_kd = torch.optim.Adam([model.kd], lr=0.001)
        for _ in range(150000):  
            optimizer_kd.zero_grad()
            output = model(L, I, I0, Imax)
            loss = nn.MSELoss()(output, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            optimizer_kd.step()
        print(f'File {filename}, Iteration {_}, Loss {loss.item()}')
        
        print(f'For file {filename}, kd={torch.abs(model.kd).item()}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.metrics import r2_score

# 三个文件路径
file_paths = [
    "E:/Desktop/mamm/403_pre.csv",
    "E:/Desktop/mamm/404_pre.csv",
    "E:/Desktop/mamm/405_pre.csv",
    "E:/Desktop/mamm/406_pre.csv",
    "E:/Desktop/mamm/407_pre.csv",
    "E:/Desktop/mamm/408_pre.csv",
    "E:/Desktop/mamm/409_pre.csv",
    "E:/Desktop/mamm/410_pre.csv",
    "E:/Desktop/mamm/412_pre.csv",
    "E:/Desktop/mamm/413_pre.csv",
    "E:/Desktop/mamm/414_pre.csv",
    "E:/Desktop/mamm/415_pre.csv",
    "E:/Desktop/mamm/540_pre.csv",
    "E:/Desktop/mamm/541_pre.csv",
    "E:/Desktop/mamm/542_pre.csv",
]

# 合并数据
all_exp, all_pre = [], []

for path in file_paths:
    df = pd.read_csv(path)
    all_exp.extend(df['RPU'].values)
    all_pre.extend(df['P1_fit'].values)

# 转为 numpy 数组
exp = np.array(all_exp)
pre = np.array(all_pre)

# 去除 <=0 的值以避免 log 错误
valid = (exp > 0) & (pre > 0)
exp = exp[valid]
pre = pre[valid]

# 计算对数
exp_log = np.log10(exp)
pre_log = np.log10(pre)

# 在对数空间中进行回归拟合
slope, intercept, r_value, p_value, std_err = linregress(exp_log, pre_log)
r_squared_log = r_value**2

# 原始空间 R²
r2_raw = r2_score(exp, pre)

# 可视化（log-log 空间）
fig, ax = plt.subplots(figsize=(5.08, 5.08))
ax.scatter(exp, pre, s=10, alpha=0.7)

# 绘制回归线 - 在对数空间中计算，然后转换回原始空间显示
x_range_log = np.linspace(min(exp_log), max(exp_log), 100)
y_range_log = slope * x_range_log + intercept
x_range = 10**x_range_log
y_range = 10**y_range_log
ax.plot(x_range, y_range, color='green', linewidth=2, label='Regression line')

# 绘制y=x参考线
min_val = min(np.min(exp), np.min(pre))
max_val = max(np.max(exp), np.max(pre))
x_ref = np.logspace(np.log10(min_val), np.log10(max_val), 100)
ax.plot(x_ref, x_ref, 'r--', linewidth=1.5, label='y=x')

ax.set_xlabel("Experimental RPU", fontsize=10)
ax.set_ylabel("Predicted RPU", fontsize=10)
ax.set_title('All Data Comparison', fontsize=12)
ax.set_xscale("log")
ax.set_yscale("log")

# 添加图例
ax.legend(fontsize=8)

# 标注
ax.annotate(f"R² = {r_squared_log:.4f}", xy=(0.05, 0.85), xycoords='axes fraction', fontsize=10)

# 美化边框
for spine in ax.spines.values():
    spine.set_linewidth(0.5)

plt.tight_layout()
plt.savefig("E:/Desktop/mamm/all_R².svg")
plt.show()

print(f"原始空间 R²: {r2_raw:.4f}")
print(f"对数空间 R²: {r_squared_log:.4f}")